In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

data = pd.read_csv('mushrooms.csv')
data.head()

data.info()



<class 'pandas.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   class                     8124 non-null   str  
 1   cap-shape                 8124 non-null   str  
 2   cap-surface               8124 non-null   str  
 3   cap-color                 8124 non-null   str  
 4   bruises                   8124 non-null   str  
 5   odor                      8124 non-null   str  
 6   gill-attachment           8124 non-null   str  
 7   gill-spacing              8124 non-null   str  
 8   gill-size                 8124 non-null   str  
 9   gill-color                8124 non-null   str  
 10  stalk-shape               8124 non-null   str  
 11  stalk-root                8124 non-null   str  
 12  stalk-surface-above-ring  8124 non-null   str  
 13  stalk-surface-below-ring  8124 non-null   str  
 14  stalk-color-above-ring    8124 non-null   str  
 15

In [2]:
data.describe().transpose()


,count,unique,top,freq
class,8124,2,e,4208
cap-shape,8124,6,x,3656
cap-surface,8124,4,y,3244
cap-color,8124,10,n,2284
bruises,8124,2,f,4748
odor,8124,9,n,3528
gill-attachment,8124,2,f,7914
gill-spacing,8124,2,c,6812
gill-size,8124,2,b,5612
gill-color,8124,12,b,1728


In [3]:
X= data.drop('class',axis=1)
X=pd.get_dummies(X,drop_first=True)
y=pd.get_dummies(data['class'],drop_first=True)
y = y.fillna({
    'False':0,
    'True':1
}).astype('int')

y

,p
0,1
1,0
2,0
3,1
4,0
...,...
8119,0
8120,0
8121,0
8122,1


In [4]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=101,test_size=0.3)

In [5]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score


xgb_model = XGBClassifier()

xgb_model.fit(X_train,y_train)

train_pred = xgb_model.predict(X_train)
test_pred =xgb_model.predict(X_test)

print('Train Accuracy :', accuracy_score(y_train, train_pred))
print('Test Accuracy :', accuracy_score(y_test, test_pred))


score = cross_val_score(estimator=xgb_model, X = X_train, y = y_train, cv = 5)
print(score.mean())


Train Accuracy : 1.0
Test Accuracy : 1.0
1.0


In [6]:
confusion_matrix(y_test,test_pred)


array([[1274,    0],
       [   0, 1164]])

In [7]:
print(classification_report(y_test,test_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1274
           1       1.00      1.00      1.00      1164

    accuracy                           1.00      2438
   macro avg       1.00      1.00      1.00      2438
weighted avg       1.00      1.00      1.00      2438



## **Hyperparameter Tuning**

In [8]:
from sklearn.model_selection import GridSearchCV 
estimator= XGBClassifier()

param_grid = {'n_estimators': [1,5,8,10,15,40,100],'max_depth':[3,4,5],'gamma':[0,0.1,0.2,.3,1,2,3]}

hp=GridSearchCV(estimator,param_grid,cv=5)
hp.fit(X_train,y_train)
hp.best_params_

{'gamma': 0, 'max_depth': 3, 'n_estimators': 40}

## **Final Model**

In [9]:
xgb= XGBClassifier(n_estimators=100,max_depth=3)
xgb.fit(X_train,y_train)

test_prediction= xgb.predict(X_test)
train_prediction = xgb.predict(X_train)

print(accuracy_score(y_train,train_prediction))
print(accuracy_score(y_test,test_prediction))

score= cross_val_score(xgb,X,y,cv=5)
print(score)
score.mean()


1.0
1.0
[0.84246154 1.         0.99753846 1.         0.81034483]


np.float64(0.9300689655172413)

In [10]:
a=xgb.feature_importances_
a

array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       2.45802294e-05, 0.00000000e+00, 9.83305363e-05, 7.18182037e-06,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 6.14007586e-04,
       1.95480455e-02, 2.58239210e-02, 2.84855650e-03, 2.91486438e-02,
       1.19454525e-01, 5.24364552e-03, 1.58385843e-01, 1.88806169e-02,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 5.52534359e-03,
       1.05777411e-02, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 9.99921685e-05, 0.00000000e+00,
       3.13461991e-04, 3.05431226e-04, 3.21328491e-01, 0.00000000e+00,
       1.86390385e-01, 1.49141136e-03, 1.37115378e-04, 0.00000000e+00,
       0.00000000e+00, 3.25704204e-05, 2.67255064e-02, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
      

In [11]:
imp_feature=pd.DataFrame(index=X.columns,data=a,columns=['feature importance'])
imp_feature

,feature importance
cap-shape_c,0.000000
cap-shape_f,0.000000
cap-shape_k,0.000000
cap-shape_s,0.000000
cap-shape_x,0.000025
...,...
habitat_l,0.000000
habitat_m,0.000000
habitat_p,0.000000
habitat_u,0.000000


In [12]:
b=imp_feature.sort_values('feature importance')

In [13]:
imp_feature=imp_feature[imp_feature['feature importance']>0.000527]
imp_feature

,feature importance
cap-color_w,0.000614
cap-color_y,0.019548
bruises_t,0.025824
odor_c,0.002849
odor_f,0.029149
odor_l,0.119455
odor_m,0.005244
odor_n,0.158386
odor_p,0.018881
gill-spacing_w,0.005525


In [14]:
imp_feature.index

Index(['cap-color_w', 'cap-color_y', 'bruises_t', 'odor_c', 'odor_f', 'odor_l',
       'odor_m', 'odor_n', 'odor_p', 'gill-spacing_w', 'gill-size_n',
       'stalk-root_c', 'stalk-root_r', 'stalk-surface-above-ring_k',
       'stalk-surface-below-ring_y', 'stalk-color-below-ring_n',
       'stalk-color-below-ring_y', 'ring-number_t', 'spore-print-color_n',
       'spore-print-color_r', 'spore-print-color_u', 'spore-print-color_w',
       'population_c', 'population_y'],
      dtype='str')

## **Final model with imp features**

In [15]:
X=X[['cap-color_w', 'cap-color_y', 'bruises_t', 'odor_c', 'odor_f', 'odor_l',
       'odor_m', 'odor_n', 'odor_p', 'gill-spacing_w', 'gill-size_n',
       'stalk-root_c', 'stalk-root_r', 'stalk-surface-above-ring_k',
       'stalk-surface-below-ring_y', 'stalk-color-below-ring_n',
       'stalk-color-below-ring_y', 'ring-number_t', 'spore-print-color_n',
       'spore-print-color_r', 'spore-print-color_u', 'spore-print-color_w',
       'population_c', 'population_y']]


In [16]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=101,test_size=0.3)

In [17]:
from xgboost import XGBClassifier
xgb= XGBClassifier(n_estimators=100,max_depth=3,gamma=0)
xgb.fit(X_train,y_train)

test_prediction= xgb.predict(X_test)
train_prediction = xgb.predict(X_train)
print('*'*20)
from sklearn.metrics import accuracy_score,confusion_matrix

print(accuracy_score(y_train,train_prediction))
print(accuracy_score(y_test,test_prediction))
print('*'*20)

score= cross_val_score(xgb,X,y,cv=5)
print(score)
print(score.mean())

print('*'*20)

print(classification_report(y_test,test_prediction))

print('*'*20)


********************
1.0
1.0
********************
[0.84246154 1.         1.         1.         0.77894089]
0.9242804850322092
********************
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1274
           1       1.00      1.00      1.00      1164

    accuracy                           1.00      2438
   macro avg       1.00      1.00      1.00      2438
weighted avg       1.00      1.00      1.00      2438

********************


In [18]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test,test_prediction)

array([[1274,    0],
       [   0, 1164]])